In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib

matplotlib.use("TkAgg")

from preprocesare import redimensionare, Imagini, centrare_date
from matematica import svd


def calculeaza_sosia_dupa_selectie(imagine_de_la_interfata, k_componente):
    if imagine_de_la_interfata is None:
        return None

    img = Image.fromarray(imagine_de_la_interfata).convert("L")
    IMG = redimensionare(np.array(img), linii, coloane)

    v_tu = IMG.flatten().astype(np.float64)
    if np.linalg.norm(v_tu) == 0:
        return None
    v_tu_normat = v_tu / np.linalg.norm(v_tu)
    v_tu_centrat = v_tu_normat - Fata_medie

    k = int(k_componente)
    U_redus = U[:, :k]
    W_redus = W[:k, :]

    w_tu = U_redus.T @ v_tu_centrat
    distante = np.linalg.norm(W_redus - w_tu[:, np.newaxis], axis=0)
    index_minim = np.argmin(distante)

    sosia_vector = X[:, index_minim]
    sosia_matrice = sosia_vector.reshape(linii, coloane)
    sosia_0_255 = (
        (sosia_matrice - sosia_matrice.min())
        / (sosia_matrice.max() - sosia_matrice.min())
        * 255
    ).astype(np.uint8)

    return sosia_0_255

Fete, n, linii, coloane, target, target_names = Imagini()

baza_de_date = []
def construieste_amprenta_faciala(imagine_din_interfata, nr_componente=50):
    if imagine_din_interfata is None:
        return None
    img = Image.fromarray(imagine_din_interfata).convert("L")
    img_redimensionata = redimensionare(np.array(img), linii, coloane)
    v_fata = img_redimensionata.flatten().astype(np.float64)
    if np.linalg.norm(v_fata) == 0:
        return None
    v_Fata_normat = v_fata / np.linalg.norm(v_fata)
    v_fata_centrat = v_Fata_normat - Fata_medie
    U_redus = U[:, :nr_componente]
    w_amprenta = U_redus.T @ v_fata_centrat
    return w_amprenta

def register(poza_fata, poza_dreapta,poza_stanga):
    global baza_de_date
    if(poza_fata is None or poza_dreapta is None or poza_stanga is None):
        return "Toate cele trei poze sunt necesare pentru înregistrare!"
    w_fata = construieste_amprenta_faciala(poza_fata, nr_componente=50)
    w_dreapta = construieste_amprenta_faciala(poza_dreapta, nr_componente=50)
    w_stanga = construieste_amprenta_faciala(poza_stanga, nr_componente=50)
    baza_de_date.append((w_fata, w_dreapta, w_stanga))
    return "Inregistrare reușită!"

def login(poza_noua_camera):
    if(poza_noua_camera is None):
        return "Poza nouă este necesară pentru autentificare!"
    if(len(baza_de_date) == 0):
        return "Baza de date este goală! Vă rugăm să vă înregistrați mai întâi."
    w_poza_camera = construieste_amprenta_faciala(poza_noua_camera, nr_componente=50)
    if w_poza_camera is None:
        return "Eroare la procesarea pozei de la cameră!"
    w_salvat_fata, w_salvat_dreapta, w_salvat_stanga = baza_de_date[-1]
    distanta_fata = np.linalg.norm(w_poza_camera - w_salvat_fata)
    distanta_dreapta = np.linalg.norm(w_poza_camera - w_salvat_dreapta)
    distanta_stanga = np.linalg.norm(w_poza_camera - w_salvat_stanga)
    if(min(distanta_fata, distanta_dreapta, distanta_stanga) < 0.5):
        return "Autentificare reușită!"
    else:
        return "Autentificare eșuată! Poza nu se potrivește cu niciuna din cele înregistrate de catre utilizator."


MAX_PER_PERSOANA = 5
selectati = []
contor = {}
for i in range(n):
    persoana = target[i]
    if persoana not in contor:
        contor[persoana] = 0
    if contor[persoana] < MAX_PER_PERSOANA:
        selectati.append(i)
        contor[persoana] += 1
    if len(selectati) >= 250:
        break

ok = 0
X_list = []
target_selectati = []
for i in selectati:
    if linii > 200 or coloane > 200:
        img_prelucrata = redimensionare(Fete[i], 200, 200)
        ok = 1
    else:
        img_prelucrata = Fete[i]
    v_brut = img_prelucrata.flatten().astype(np.float64)
    if np.linalg.norm(v_brut) == 0:
        continue
    v_normat = v_brut / np.linalg.norm(v_brut)
    X_list.append(v_normat)
    target_selectati.append(target[i])

X = np.array(X_list).T
if ok == 1:
    linii, coloane = 200, 200
A, Fata_medie = centrare_date(X)

"""
plt.imshow(Fata_medie.reshape(linii, coloane), cmap='gray')
plt.show()
"""

U, S, Vt = svd(A)

"""
Afișăm prima Eigenface (cea mai importantă)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(U[:, 0].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 1 (Cea mai mare valoare proprie)")

Afișăm a doua Eigenface (următoarea ca importanță)
plt.subplot(1, 2, 2)
plt.imshow(U[:, 1].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 2")
plt.show()
"""

W = U.T @ A
"""
# 1. Extragem ponderile primei fețe (prima coloană din W)
w1 = W[:, 0] 
# 2. Reconstruim fața în spațiul pixelilor
# Înmulțim matricea U (Eigenfaces) cu vectorul de ponderi w1
fata_reconstruita_centrata =  U[:, :100] @ w1
# 3. Adăugăm înapoi Fața Medie (psi) pentru a reveni la aspectul original
fata_finala = fata_reconstruita_centrata + Fata_medie.flatten()
# 4. Afișăm rezultatul
plt.imshow(fata_finala.reshape(linii, coloane), cmap='gray')
plt.title("Prima față reconstruită din ponderile W")
plt.show() 
"""

custom_theme = gr.themes.Monochrome(
    primary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    radius_size=gr.themes.sizes.radius_lg,
).set(
    body_background_fill="#000000",
    body_background_fill_dark="#000000",
    body_text_color="white",
    body_text_color_dark="white",
    background_fill_primary="#050505",
    background_fill_primary_dark="#050505",
    background_fill_secondary="#0a0a0a",
    background_fill_secondary_dark="#0a0a0a",
    block_background_fill="rgba(255, 255, 255, 0.03)",
    block_background_fill_dark="rgba(255, 255, 255, 0.03)",
    block_border_width="1px",
    block_border_color="rgba(255, 255, 255, 0.08)",
    block_border_color_dark="rgba(255, 255, 255, 0.08)",
    button_primary_background_fill="white",
    button_primary_background_fill_dark="white",
    button_primary_text_color="black",
    button_primary_text_color_dark="black",
    shadow_drop="0px 4px 20px rgba(0, 0, 0, 0.5)",
)

custom_css = """
body, .gradio-container {
    background: radial-gradient(circle at 50% 20%, #152220 0%, #000000 70%) !important;
    color: #ffffff !important;
}
.tab-nav {
    background: rgba(255, 255, 255, 0.05) !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 9999px !important;
    padding: 6px !important;
    display: flex;
    justify-content: center;
    gap: 10px;
    margin: 0 auto 30px auto !important;
    width: fit-content !important;
}
.tab-nav button {
    border-radius: 9999px !important;
    border: none !important;
    padding: 8px 24px !important;
    font-weight: 500 !important;
    color: #aaaaaa !important;
    transition: all 0.3s ease;
}
.tab-nav button.selected {
    background: rgba(255, 255, 255, 0.1) !important;
    color: white !important;
}
.gr-box {
    backdrop-filter: blur(15px);
}
h1 {
    text-align: center;
    font-size: 3.5em !important;
    font-weight: 600 !important;
    letter-spacing: -0.03em;
    margin-top: 1em !important;
    margin-bottom: 0.5em !important;
}
p {
    text-align: center;
    color: #aaaaaa;
}
"""

with gr.Blocks(theme=custom_theme, css=custom_css, title="Sistem Complet Recunoaștere Facială (SVD)") as app_autentificare:
    gr.Markdown("# One-click for Face Defense")
    
    with gr.Tab("1. Găsește Sosia (1:N)"):
        gr.Markdown("Încarcă o poză și reglează parametrul de componente principale (k) pentru a găsi fața cea mai asemănătoare din baza de date publică.")
        with gr.Row():
            with gr.Column():
                img_sosie_in = gr.Image(label="1. Alege/Trage poza ta aici")
                k_slider = gr.Slider(minimum=1, maximum=150, value=15, step=1, label="Număr Componente Principale (k)")
                btn_sosie = gr.Button("Calculează Sosia", variant="primary")
            with gr.Column():
                img_sosie_out = gr.Image(label="2. Sosia ta calculată", image_mode="L")
        
        btn_sosie.click(fn=calculeaza_sosia_dupa_selectie, inputs=[img_sosie_in, k_slider], outputs=img_sosie_out)
        
    with gr.Tab("2. Înrolează-te (Enroll)"):
        gr.Markdown("Înainte de a folosi sistemul de securitate, algoritmul trebuie să te învețe. Fă **3 capturi (frontal, stânga, dreapta)** folosind camera web sau încarcă poze de pe PC.")
        
        with gr.Row():
            poza1 = gr.Image(label="Vedere Frontală")
            poza2 = gr.Image(label="Vedere Stânga")
            poza3 = gr.Image(label="Vedere Dreapta")
            
        btn_inreg = gr.Button("Vreau să mă înregistrez în Sistem!", variant="primary")
        rez_inreg = gr.Textbox(label="Status Înregistrare")
        
        btn_inreg.click(fn=register, inputs=[poza1, poza2, poza3], outputs=rez_inreg)
        
    with gr.Tab("3. Autentifică-te (Login)"):
        gr.Markdown("Încarcă o poză pentru a testa dacă bariera de securitate te recunoaște. Se va returna 'Autentificare reușită' dacă ești înregistrat.")
        
        poza_login = gr.Image(label="Camera Securitate")
        btn_login = gr.Button("Verifică Identitatea", variant="primary")
        rez_login = gr.Textbox(label="Decizie Sistem")
        
        btn_login.click(fn=login, inputs=poza_login, outputs=rez_login)

app_autentificare.launch()

/tmp/ipykernel_118919/1516239693.py:231: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Sistem Complet Recunoaștere Facială (SVD)") as app_autentificare:


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
